# EDA: QCEW Growth

Purpose: inspect national QCEW growth coverage, sector mix, growth-score distributions, and latest-period opportunity signals before Tableau/dashboarding.

## Setup

Install if needed:

```bash
pip install pandas numpy matplotlib snowflake-connector-python
```

For browser authentication:

```bash
export SNOWFLAKE_ACCOUNT='<account_locator>'
export SNOWFLAKE_USER='<username>'
export SNOWFLAKE_AUTHENTICATOR='externalbrowser'
```

For password authentication, use `SNOWFLAKE_PASSWORD` instead of `SNOWFLAKE_AUTHENTICATOR`.

In [ ]:

from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

DATABASE = "SMB_MARKET_INTELLIGENCE_DEV"
WAREHOUSE = os.getenv("SNOWFLAKE_WAREHOUSE", "COMPUTE_WH")
ROLE = os.getenv("SNOWFLAKE_ROLE", "ACCOUNTADMIN")

def get_connection():
    """Create a Snowflake connection from environment variables.

    Required environment variables for password auth:
        SNOWFLAKE_ACCOUNT
        SNOWFLAKE_USER
        SNOWFLAKE_PASSWORD

    Optional:
        SNOWFLAKE_AUTHENTICATOR=externalbrowser
        SNOWFLAKE_WAREHOUSE=COMPUTE_WH
        SNOWFLAKE_ROLE=ACCOUNTADMIN
    """
    import snowflake.connector

    account = os.getenv("SNOWFLAKE_ACCOUNT")
    user = os.getenv("SNOWFLAKE_USER")
    password = os.getenv("SNOWFLAKE_PASSWORD")
    authenticator = os.getenv("SNOWFLAKE_AUTHENTICATOR")

    if not account or not user:
        raise RuntimeError(
            "Set SNOWFLAKE_ACCOUNT and SNOWFLAKE_USER before running this notebook. "
            "For browser auth, also set SNOWFLAKE_AUTHENTICATOR=externalbrowser. "
            "For password auth, set SNOWFLAKE_PASSWORD."
        )

    kwargs = {
        "account": account,
        "user": user,
        "warehouse": WAREHOUSE,
        "database": DATABASE,
        "role": ROLE,
    }

    if authenticator:
        kwargs["authenticator"] = authenticator
    elif password:
        kwargs["password"] = password
    else:
        raise RuntimeError(
            "Set either SNOWFLAKE_PASSWORD or SNOWFLAKE_AUTHENTICATOR=externalbrowser."
        )

    return snowflake.connector.connect(**kwargs)

def read_sql(query):
    with get_connection() as conn:
        return pd.read_sql(query, conn)

def show_basic_profile(df):
    display(df.head())
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns):,}")


## Load QCEW growth scores

In [ ]:

qcew = read_sql("""
SELECT
    state_fips,
    state_abbr,
    state_name,
    county_fips,
    qcew_industry_code,
    sector_name,
    year,
    quarter,
    period_id,
    qtrly_estabs,
    avg_monthly_employment,
    total_qtrly_wages,
    avg_wkly_wage,
    oty_qtrly_estabs_pct_chg,
    oty_total_qtrly_wages_pct_chg,
    ongoing_growth_score,
    expected_growth_score,
    ongoing_growth_tier,
    expected_growth_tier
FROM INT.INT_QCEW_GROWTH_SCORES
""")

qcew.columns = qcew.columns.str.lower()
show_basic_profile(qcew)


## Coverage by period

In [ ]:

coverage = (
    qcew.groupby(["year", "quarter", "period_id"])
    .agg(
        rows=("county_fips", "size"),
        states=("state_fips", "nunique"),
        counties=("county_fips", "nunique"),
        sectors=("qcew_industry_code", "nunique"),
        establishments=("qtrly_estabs", "sum"),
        employment=("avg_monthly_employment", "sum"),
        wages=("total_qtrly_wages", "sum"),
    )
    .reset_index()
    .sort_values(["year", "quarter"])
)

display(coverage.tail(12))


In [ ]:

plt.figure(figsize=(9, 4.5))
plt.plot(coverage["period_id"], coverage["rows"], marker="o")
plt.title("QCEW rows by period")
plt.xlabel("Period")
plt.ylabel("Rows")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Latest-period sector mix

In [ ]:

latest_period = qcew["period_id"].max()
latest = qcew[qcew["period_id"] == latest_period].copy()

sector_mix = (
    latest.groupby(["qcew_industry_code", "sector_name"])
    .agg(
        rows=("county_fips", "size"),
        counties=("county_fips", "nunique"),
        establishments=("qtrly_estabs", "sum"),
        employment=("avg_monthly_employment", "sum"),
        wages=("total_qtrly_wages", "sum"),
        avg_ongoing_growth_score=("ongoing_growth_score", "mean"),
        avg_expected_growth_score=("expected_growth_score", "mean"),
    )
    .reset_index()
    .sort_values("establishments", ascending=False)
)

display(sector_mix)


In [ ]:

plt.figure(figsize=(9, 4.5))
plt.bar(sector_mix["sector_name"], sector_mix["establishments"])
plt.title(f"Latest-period establishments by sector: {latest_period}")
plt.xlabel("Sector")
plt.ylabel("Establishments")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## Growth score distributions

In [ ]:

for score_col in ["ongoing_growth_score", "expected_growth_score"]:
    plt.figure(figsize=(8, 4.5))
    plt.hist(latest[score_col].dropna(), bins=30)
    plt.title(f"{score_col} distribution, latest period {latest_period}")
    plt.xlabel(score_col)
    plt.ylabel("County-industry rows")
    plt.tight_layout()
    plt.show()


## Top latest-period growth markets

In [ ]:

top_growth = (
    latest.sort_values(["expected_growth_score", "ongoing_growth_score"], ascending=False)
    .loc[
        :,
        [
            "state_abbr",
            "county_fips",
            "sector_name",
            "qtrly_estabs",
            "avg_monthly_employment",
            "oty_qtrly_estabs_pct_chg",
            "ongoing_growth_score",
            "expected_growth_score",
            "ongoing_growth_tier",
            "expected_growth_tier",
        ],
    ]
    .head(25)
)

display(top_growth)


## County concentration diagnostic

This checks whether the national opportunity base is concentrated in a small number of counties.

In [ ]:

county_latest = (
    latest.groupby(["state_abbr", "county_fips"])
    .agg(establishments=("qtrly_estabs", "sum"))
    .reset_index()
    .sort_values("establishments", ascending=False)
)

county_latest["cumulative_establishment_share"] = (
    county_latest["establishments"].cumsum() / county_latest["establishments"].sum()
)

display(county_latest.head(20))

plt.figure(figsize=(9, 4.5))
plt.plot(
    np.arange(1, len(county_latest) + 1),
    county_latest["cumulative_establishment_share"],
)
plt.title(f"Cumulative establishment share by county, {latest_period}")
plt.xlabel("County rank by target-sector establishments")
plt.ylabel("Cumulative share")
plt.tight_layout()
plt.show()


## Notes to capture

- Are all expected states and sectors present in the latest period?
- Are score distributions approximately percentile-like?
- Are any sectors or geographies unexpectedly sparse?
- Which county-industries repeatedly appear near the top?